In [28]:
%pip install openai pdf2image pillow
%pip install dotenv


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [35]:
from dotenv import load_dotenv
from openai import OpenAI
from pdf2image import convert_from_path
import base64
import os
import re

In [40]:
load_dotenv()  # read .env file
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI()
filename = "images/jee_mains_page_2.jpg"
with open(filename, "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode("utf-8")
    response = client.chat.completions.create (
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a math OCR assistant that converts scanned questions into Markdown with LaTeX."},
            {"role": "user", "content": [
                {"type": "text", "text": "Extract the math questions. Use Markdown and LaTeX for formulas."},
                {"type": "image_url", "image_url": 
                     {"url": f"data:image/png;base64,{img_b64}"}}
            ]}
        ]
    )

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************qeYA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [37]:
print(response.choices[0].message.content)

Here are the extracted math questions formatted in Markdown with LaTeX for the formulas:

```markdown
12. Let \( f^{(1)} \) denote \( g(x) \), \( f^{(2)} \) denotes \( g(g(x)) \), \( f^{(3)} \) denotes \( g(g(g(x))) \) and so on. If \( g(x) \) is an increasing function and lies between \( (0, \infty) \), \( \forall x \in \mathbb{R} \), then 
   \[
   \lim_{n \to \infty} \frac{1}{f^{(n)}(x)} 
   \]
   \((a)\) \(0\) \((b)\) Does not exist \((d)\) None of these

13. \( f(x) \) is increasing, concave up function for all \( x \in [a, b] \), if for any \( \lambda > 0 \), 
   \[
   \frac{(b + a)}{1 + \lambda} f(a) \leq \frac{f(b) + \lambda f(a)}{1 + \lambda} 
   \]
   \((a)\) \(b + a \geq 0\) \((c)\) None of these 

15. Let \( f: [2, 7] \to \mathbb{R} \) be a continuous and differentiable function. Then, the value of \( f(7) - f(2) \) is 
   \[
   f(7) = (7)^3 + f(2)^2 
   \]
   (where \( c \in (2,7) \)) 
   \((a)\) \(3f'(c)f''(c)\) \((b)\) \( 5f'(c)f''(c) \) \((c)\) \(2f^{(3)}(c)\) \((d)\) N

In [39]:
text = response.choices[0].message.content
pattern = r'(?m)^\d.*(?:\n^(?!\d).*|\n)*(?=\n\d|\Z)'
matches = re.findall(pattern, text)

with open('output/jee_mains_page_2.md', 'w') as of:
    for match in matches:
        question = ' '.join(match.splitlines())
        of.writelines(question)
        of.write('\n')